# Biomass Plant Financial Model

Model the cashflows of a biomass plant (wood chip to electricity) from Q1 2017 through Q4 2026.
Determine the purchase price for a 20% stake in the plant's cashflows.

In [ ]:
# ============================================================
# CELL 1: Imports and load data from Excel workbook
# ============================================================
import os
import re
import calendar
from datetime import date, datetime
import pandas as pd

# Determine data directory
DATA_DIR = '/workspace/data'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'environment', 'data')

XLSX_FILE = 'MO16-Round-1-Sec-2-Chip-Off-The-Old-Block.xlsx'
XLSX_PATH = os.path.join(DATA_DIR, XLSX_FILE)
print(f'Data directory: {DATA_DIR}')
print(f'Excel file exists: {os.path.exists(XLSX_PATH)}')

# Read the Assumptions sheet to extract market prices
assumptions_df = pd.read_excel(XLSX_PATH, sheet_name='Assumptions')

# Market prices are in rows 42-52 (0-indexed), columns 4 (Start), 5 (End), 8 (Price)
mp_data = assumptions_df.iloc[42:53, [4, 5, 8]]
mp_data.columns = ['Start', 'End', 'Price']
mp_data['Start'] = pd.to_datetime(mp_data['Start'])
mp_data['End'] = pd.to_datetime(mp_data['End'])
mp_data['Price'] = mp_data['Price'].astype(float)

# Convert to list of tuples (start_date, end_date, price)
market_prices = []
for _, row in mp_data.iterrows():
    market_prices.append((row['Start'].date(), row['End'].date(), row['Price']))

print(f'\nMarket price periods: {len(market_prices)}')
for s, e, p in market_prices:
    print(f'  {s} to {e}: ${p:.0f}/MWh')

In [ ]:
# ============================================================
# CELL 2: Define model parameters from the introduction
# ============================================================

# Plant capacity and operation
capacity_mw = 9       # MW
availability = 0.95   # 95%
hours_per_day = 24

# Shutdown periods (from the start of the first day to end of final day)
shutdowns = [
    (date(2017, 6, 1),  date(2017, 6, 15)),
    (date(2019, 12, 15), date(2020, 1, 3)),
    (date(2021, 4, 5),  date(2021, 4, 25)),
    (date(2023, 10, 25), date(2023, 11, 10)),
    (date(2025, 5, 6),  date(2025, 5, 26)),
    (date(2026, 6, 26), date(2026, 7, 5)),
]

# Electricity pricing
cap_base = 70.0    # $/MWh cap price (2017 base)
cap_inf = 0.02     # 2% per annum
floor_base = 45.0  # $/MWh floor price (2017 base)
floor_inf = 0.01   # 1% per annum

# Supplier 1 (inflated)
s1_base = 100.0    # $/tonne (2017 base)
s1_inf = 0.02      # 2% per annum
s1_mwh_per_t = 3.5 # MWh per tonne
s1_max_t = 4500.0  # max tonnes per quarter

# Supplier 2 (fixed price, unlimited)
s2_price = 130.0   # $/tonne (no inflation)
s2_mwh_per_t = 4.0 # MWh per tonne

# Fixed costs
fc_base = 75000.0  # $/month (2017 base)
fc_inf = 0.015     # 1.5% per annum

# Valuation
stake = 0.20       # 20%
rate_mkt = 0.10    # 10% required return (Q1-Q12)
rate_flr = 0.04    # 4% required return (Q13)
val_date = date(2016, 12, 31)  # acquisition date

print('Parameters loaded.')
print(f'  Plant: {capacity_mw}MW at {availability*100}% availability')
print(f'  Shutdowns: {len(shutdowns)} periods')
print(f'  Cap: ${cap_base}/MWh @ {cap_inf*100}% p.a.')
print(f'  Floor: ${floor_base}/MWh @ {floor_inf*100}% p.a.')
print(f'  S1: ${s1_base}/t, {s1_mwh_per_t} MWh/t, max {s1_max_t}t/q, {s1_inf*100}% p.a.')
print(f'  S2: ${s2_price}/t, {s2_mwh_per_t} MWh/t, unlimited, fixed')
print(f'  Fixed costs: ${fc_base:,.0f}/month @ {fc_inf*100}% p.a.')

In [ ]:
# ============================================================
# CELL 3: Build quarterly financial model
# ============================================================

def quarter_bounds(year, q):
    """Return (start_date, end_date) for a given year and quarter."""
    m1 = (q - 1) * 3 + 1
    m3 = q * 3
    return date(year, m1, 1), date(year, m3, calendar.monthrange(year, m3)[1])

def count_shutdown_days(q_start, q_end):
    """Count total shutdown days overlapping with [q_start, q_end]."""
    total = 0
    for sd_s, sd_e in shutdowns:
        os_ = max(q_start, sd_s)
        oe_ = min(q_end, sd_e)
        if os_ <= oe_:
            total += (oe_ - os_).days + 1
    return total

def get_market_price(q_start):
    """Get market price for the April-March period containing q_start."""
    for mp_s, mp_e, mp in market_prices:
        if mp_s <= q_start <= mp_e:
            return mp
    raise ValueError(f'No market price for {q_start}')

# Build 40 quarters: Q1 2017 through Q4 2026
quarters = []
for year in range(2017, 2027):
    for q in range(1, 5):
        qs, qe = quarter_bounds(year, q)
        total_days = (qe - qs).days + 1
        sd_days = count_shutdown_days(qs, qe)
        op_days = total_days - sd_days
        
        # Electricity production
        mwh = op_days * hours_per_day * capacity_mw * availability
        
        # Inflation exponent (applied Jan 1 each year after 2017)
        n = year - 2017
        
        # Prices
        cap = cap_base * (1 + cap_inf) ** n
        floor = floor_base * (1 + floor_inf) ** n
        mkt = get_market_price(qs)
        eff = min(cap, max(floor, mkt))  # market clamped by floor and cap
        
        # Revenue
        rev_mkt = mwh * eff   # revenue at effective market price
        rev_flr = mwh * floor # revenue at floor price only
        
        # Wood chip costs
        s1_p = s1_base * (1 + s1_inf) ** n   # S1 price per tonne
        s1_cpm = s1_p / s1_mwh_per_t         # S1 cost per MWh
        s2_cpm = s2_price / s2_mwh_per_t     # S2 cost per MWh = 32.50
        s1_max_mwh = s1_max_t * s1_mwh_per_t # S1 max MWh = 15750
        
        # Buy from cheapest supplier first, top up with other
        if s1_cpm <= s2_cpm:
            mwh_s1 = min(mwh, s1_max_mwh)
            mwh_s2 = max(0.0, mwh - mwh_s1)
        else:
            mwh_s1 = 0.0
            mwh_s2 = mwh
        
        t_s1 = mwh_s1 / s1_mwh_per_t
        t_s2 = mwh_s2 / s2_mwh_per_t
        wood = t_s1 * s1_p + t_s2 * s2_price
        
        # Fixed costs (3 months per quarter)
        fc = 3 * fc_base * (1 + fc_inf) ** n
        
        # Net cash flows
        ncf_mkt = rev_mkt - wood - fc
        ncf_flr = rev_flr - wood - fc
        
        # Discount factors (actual/365 from val_date to end of quarter)
        days_from_base = (qe - val_date).days
        df_m = 1.0 / (1 + rate_mkt) ** (days_from_base / 365.0)
        df_f = 1.0 / (1 + rate_flr) ** (days_from_base / 365.0)
        
        quarters.append({
            'year': year, 'q': q, 'qs': qs, 'qe': qe,
            'days': total_days, 'sd': sd_days, 'op': op_days, 'mwh': mwh,
            'cap': cap, 'floor': floor, 'mkt': mkt, 'eff': eff,
            'rev_mkt': rev_mkt, 'rev_flr': rev_flr,
            's1_p': s1_p, 's1_cpm': s1_cpm, 's2_cpm': s2_cpm,
            'mwh_s1': mwh_s1, 'mwh_s2': mwh_s2,
            't_s1': t_s1, 't_s2': t_s2, 'wood': wood, 'fc': fc,
            'ncf_mkt': ncf_mkt, 'ncf_flr': ncf_flr,
            'pv_m': ncf_mkt * df_m, 'pv_f': ncf_flr * df_f,
        })

df = pd.DataFrame(quarters)

print(f'Quarterly model: {len(df)} quarters')
print(f'Total MWh:              {df.mwh.sum():>15,.1f}')
print(f'Total revenue (market): ${df.rev_mkt.sum():>15,.2f}')
print(f'Total revenue (floor):  ${df.rev_flr.sum():>15,.2f}')
print(f'Total wood cost:        ${df.wood.sum():>15,.2f}')
print(f'Total fixed costs:      ${df.fc.sum():>15,.2f}')
print(f'Net CF (market):        ${df.ncf_mkt.sum():>15,.2f}')
print(f'Net CF (floor):         ${df.ncf_flr.sum():>15,.2f}')
print(f'PV at 10% (market):     ${df.pv_m.sum():>15,.2f}')
print(f'PV at 4% (floor):       ${df.pv_f.sum():>15,.2f}')

In [ ]:
# ============================================================
# CELL 4: Compute answers and set the answers dictionary
# ============================================================

def match_mc(question_file, value):
    """Match a computed numeric value to the closest MC option."""
    with open(question_file) as f:
        text = f.read()
    options = {}
    for m in re.finditer(r'([A-I])\.\s+(.+)', text):
        letter = m.group(1)
        cleaned = m.group(2).strip().replace('$', '').replace(',', '')
        try:
            options[letter] = float(cleaned)
        except ValueError:
            options[letter] = m.group(2).strip()
    best = None
    best_dist = float('inf')
    for letter, ov in options.items():
        if isinstance(ov, (int, float)):
            d = abs(value - ov)
            if d < best_dist:
                best_dist = d
                best = letter
    return best

def match_mc_date(question_file, target_date):
    """Match a date to a MC option."""
    MN = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
          7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
    fmt = f'{target_date.day} {MN[target_date.month]} {target_date.year}'
    with open(question_file) as f:
        text = f.read()
    for m in re.finditer(r'([A-I])\.\s+(.+)', text):
        if m.group(2).strip() == fmt:
            return m.group(1)
    return None

# Q1: Operational days in Q4 2023
q1_val = int(df[(df.year == 2023) & (df.q == 4)].op.iloc[0])
q1 = match_mc(f'{DATA_DIR}/question1.txt', q1_val)
print(f'Q1:  Op days Q4 2023 = {q1_val} -> {q1}')

# Q2: Total MWh over 10 years
q2_val = df.mwh.sum()
q2 = match_mc(f'{DATA_DIR}/question2.txt', q2_val)
print(f'Q2:  Total MWh = {q2_val:,.1f} -> {q2}')

# Q3: Cap price in April 2025 (n=8)
q3_val = cap_base * (1 + cap_inf) ** 8
q3 = match_mc(f'{DATA_DIR}/question3.txt', q3_val)
print(f'Q3:  Cap price Apr 2025 = ${q3_val:.4f} -> {q3}')

# Q4: Total revenue (market with floor/cap)
q4_val = df.rev_mkt.sum()
q4 = match_mc(f'{DATA_DIR}/question4.txt', q4_val)
print(f'Q4:  Revenue (mkt) = ${q4_val:,.2f} -> {q4}')

# Q5: Total revenue (floor only)
q5_val = df.rev_flr.sum()
q5 = match_mc(f'{DATA_DIR}/question5.txt', q5_val)
print(f'Q5:  Revenue (flr) = ${q5_val:,.2f} -> {q5}')

# Q6: When S2 first becomes cheaper per MWh
q6_row = df[df.s1_cpm > df.s2_cpm].iloc[0]
q6_date = q6_row.qs
q6 = match_mc_date(f'{DATA_DIR}/question6.txt', q6_date)
print(f'Q6:  S2 cheaper from {q6_date} -> {q6}')

# Q7: Wood chip cost Q1 2018
q7_val = df[(df.year == 2018) & (df.q == 1)].wood.iloc[0]
q7 = match_mc(f'{DATA_DIR}/question7.txt', q7_val)
print(f'Q7:  Wood cost Q1 2018 = ${q7_val:,.2f} -> {q7}')

# Q8: Wood chip cost Q3 2023
q8_val = df[(df.year == 2023) & (df.q == 3)].wood.iloc[0]
q8 = match_mc(f'{DATA_DIR}/question8.txt', q8_val)
print(f'Q8:  Wood cost Q3 2023 = ${q8_val:,.2f} -> {q8}')

# Q9: Total tonnes from S2 over 10 years
q9_val = df.t_s2.sum()
q9 = match_mc(f'{DATA_DIR}/question9.txt', q9_val)
print(f'Q9:  Tonnes S2 = {q9_val:,.1f} -> {q9}')

# Q10: Net cash flow (market)
q10_val = df.ncf_mkt.sum()
q10 = match_mc(f'{DATA_DIR}/question10.txt', q10_val)
print(f'Q10: NCF (mkt) = ${q10_val:,.2f} -> {q10}')

# Q11: Net cash flow (floor)
q11_val = df.ncf_flr.sum()
q11 = match_mc(f'{DATA_DIR}/question11.txt', q11_val)
print(f'Q11: NCF (flr) = ${q11_val:,.2f} -> {q11}')

# Q12: Purchase price 20% stake at 10% (market)
q12_val = round(stake * df.pv_m.sum())
print(f'Q12: Price (10%, mkt) = ${q12_val:,}')

# Q13: Purchase price 20% stake at 4% (floor)
q13_val = round(stake * df.pv_f.sum())
print(f'Q13: Price (4%, flr) = ${q13_val:,}')

# ============================================================
# SET THE ANSWERS DICTIONARY
# ============================================================
answers = {
    'question1': q1,
    'question2': q2,
    'question3': q3,
    'question4': q4,
    'question5': q5,
    'question6': q6,
    'question7': q7,
    'question8': q8,
    'question9': q9,
    'question10': q10,
    'question11': q11,
    'question12': q12_val,
    'question13': q13_val,
}

print('\n' + '=' * 50)
print('FINAL ANSWERS')
print('=' * 50)
for k in sorted(answers, key=lambda x: int(x.replace('question',''))):
    print(f'  {k}: {answers[k]}')
print('=' * 50)